# 10 — Human in the Loop

        **Mode:** `live`  
        **Version:** Praval `0.8.0`  
        **Course link:** New in Praval 0.8

        This notebook makes the framework's execution visible. It records actual
        stages and renders the messages or runtime events that connect them.

        **You will see**

        - A real model originates a guarded tool call.
- Praval persists a pending intervention before execution.
- Approve, edit, and reject decisions resume through the same run state.

        Run the cells from top to bottom. Committed notebooks contain no saved
        output, so everything shown is produced by your installed Praval package.


In [ ]:
from __future__ import annotations

import html as _html
import json
import os
import time
from pathlib import Path

from IPython.display import HTML, display

os.environ.setdefault("PRAVAL_OBSERVABILITY", "off")


from praval import get_provider_registry
from praval.models import ModelResponse, ProviderCapabilities


class NotebookLifecycleProvider:
    """Credential-free adapter for agents whose handlers do not call a model."""

    provider_name = "notebook-lifecycle"
    capabilities = ProviderCapabilities(text=True)

    def __init__(self, config):
        self.config = config

    def invoke(self, request, tools=None):
        return ModelResponse(
            content="Notebook lifecycle response",
            provider=self.provider_name,
            model=request.model,
        )

    def close(self):
        return None


_notebook_registry = get_provider_registry()
_notebook_registry.register_provider(
    "notebook-lifecycle",
    NotebookLifecycleProvider,
    default_model="notebook-lifecycle-model",
    capabilities=NotebookLifecycleProvider.capabilities,
)
os.environ.setdefault("PRAVAL_DEFAULT_PROVIDER", "notebook-lifecycle")
os.environ.setdefault("PRAVAL_DEFAULT_MODEL", "notebook-lifecycle-model")

EVENTS = []
_STARTED = time.perf_counter()


def stage(actor, action, detail="", spore=None):
    """Capture one observable execution stage."""
    EVENTS.append(
        {
            "ms": round((time.perf_counter() - _STARTED) * 1000, 1),
            "actor": actor,
            "action": action,
            "detail": detail,
            "spore_id": getattr(spore, "id", "") if spore else "",
        }
    )


def show_flow(*steps):
    cards = []
    for index, (name, detail) in enumerate(steps):
        if index:
            cards.append('<div style="font-size:24px;color:#557">→</div>')
        cards.append(
            '<div style="padding:12px 16px;border:1px solid #b8c7dc;'
            'border-radius:12px;background:#f7fbff;min-width:130px">'
            f'<b>{_html.escape(name)}</b><br>'
            f'<span style="color:#456;font-size:12px">'
            f'{_html.escape(detail)}</span></div>'
        )
    display(
        HTML(
            '<div style="display:flex;gap:10px;align-items:center;'
            'flex-wrap:wrap;margin:12px 0">' + "".join(cards) + "</div>"
        )
    )


def show_timeline(events=None):
    events = EVENTS if events is None else events
    rows = []
    for item in events:
        rows.append(
            "<tr>"
            f"<td>{item['ms']:.1f}</td>"
            f"<td><b>{_html.escape(str(item['actor']))}</b></td>"
            f"<td>{_html.escape(str(item['action']))}</td>"
            f"<td>{_html.escape(str(item['detail']))}</td>"
            f"<td><code>{_html.escape(str(item['spore_id'])[:12])}</code></td>"
            "</tr>"
        )
    display(
        HTML(
            '<table style="border-collapse:collapse;width:100%">'
            '<thead><tr><th>ms</th><th>actor</th><th>stage</th>'
            '<th>detail</th><th>spore</th></tr></thead><tbody>'
            + "".join(rows)
            + "</tbody></table>"
        )
    )


def show_spore(spore, label="Spore on the Reef"):
    payload = json.loads(spore.to_json())
    rendered = _html.escape(json.dumps(payload, indent=2, sort_keys=True))
    display(
        HTML(
            '<div style="border-left:5px solid #7b61ff;padding:10px 14px;'
            'background:#faf9ff;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def show_json(value, label="Result"):
    rendered = _html.escape(json.dumps(value, indent=2, sort_keys=True, default=str))
    display(
        HTML(
            '<div style="border-left:5px solid #18a999;padding:10px 14px;'
            'background:#f5fffd;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def require_env(*names):
    missing = [name for name in names if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Missing required notebook configuration: " + ", ".join(missing))
    return {name: os.environ[name] for name in names}


def find_example_asset(relative):
    relative = Path(relative)
    starts = [Path.cwd(), *Path.cwd().parents]
    for root in starts:
        for candidate in (root / relative, root / "examples" / relative):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f"Could not locate example asset: {relative}")


## Live approval prerequisite

This notebook requires OpenAI because the tool call must originate from a real
model. Set `PRAVAL_NOTEBOOK_HITL_DECISION` to `approve`, `edit`, or `reject`.
The default is `approve`; `edit` changes the tool arguments before execution.


In [ ]:
import tempfile

values = require_env("OPENAI_API_KEY", "PRAVAL_OPENAI_MODEL")
decision = os.environ.get("PRAVAL_NOTEBOOK_HITL_DECISION", "approve").lower()
if decision not in {"approve", "edit", "reject"}:
    raise RuntimeError("PRAVAL_NOTEBOOK_HITL_DECISION must be approve, edit, or reject")
workspace = tempfile.TemporaryDirectory(prefix="praval-hitl-notebook-")
database = str(Path(workspace.name) / "hitl.sqlite3")
show_flow(
    ("Model", "tool call"),
    ("Praval", "persist pending run"),
    ("Human", decision),
    ("Runtime", "resume"),
)


In [ ]:
from praval import Agent, InterventionRequired, ToolSpec

executions = []
agent = Agent(
    "course-hitl-agent",
    provider="openai",
    model=values["PRAVAL_OPENAI_MODEL"],
    hitl_enabled=True,
    hitl_db_path=database,
    config={"temperature": 0, "max_output_tokens": 96, "timeout": 60},
)


def guarded_multiply(a: int, b: int) -> int:
    executions.append({"a": a, "b": b, "result": a * b})
    stage("tool", "executed", f"{a} × {b}")
    return a * b


agent.add_tool_spec(
    ToolSpec(
        name="guarded_multiply",
        description="Multiply two integers; always use this for multiplication.",
        parameters={
            "type": "object",
            "properties": {
                "a": {"type": "integer"},
                "b": {"type": "integer"},
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        },
        strict=True,
        requires_approval=True,
        risk_level="high",
        approval_reason="The notebook operator must approve this execution.",
    ),
    guarded_multiply,
)


In [ ]:
try:
    agent.generate(
        "You must call guarded_multiply with a=2 and b=3. "
        "Do not calculate the answer yourself."
    )
except InterventionRequired as interruption:
    stage("HITL", "interrupted", interruption.tool_name)
    intervention_state = {
        "agent": interruption.agent_name,
        "tool": interruption.tool_name,
        "arguments": interruption.arguments,
        "intervention_id": interruption.intervention_id,
        "run_id": interruption.run_id,
        "decision": decision,
    }
else:
    raise AssertionError("The real model did not produce the required tool call")

assert executions == []
show_json(intervention_state, "Persisted pending intervention")


In [ ]:
if decision == "approve":
    agent.approve_intervention(
        intervention_state["intervention_id"], reviewer="notebook-operator"
    )
elif decision == "edit":
    agent.approve_intervention(
        intervention_state["intervention_id"],
        reviewer="notebook-operator",
        edited_args={"a": 7, "b": 8},
    )
else:
    agent.reject_intervention(
        intervention_state["intervention_id"],
        reviewer="notebook-operator",
        reason="Demonstrating the reject path",
    )
stage("human", decision, "intervention recorded")
resumed = agent.resume_run(intervention_state["run_id"])
stage("runtime", "resumed", f"{len(resumed)} response chars")

if decision == "approve":
    assert executions[0]["result"] == 6
elif decision == "edit":
    assert executions[0] == {"a": 7, "b": 8, "result": 56}
else:
    assert executions == []
show_json({"response": resumed, "executions": executions}, "Resumed run")
show_timeline()


In [ ]:
agent.close()
workspace.cleanup()
stage("agent", "closed", "HITL database removed")
